# Phase 3 — Bar Agent with Profile-Aware Publishing

This notebook runs one bar simulation context and publishes data for centralized matching.

Phase 3 scope:
- Local movement and conversation logic
- Profile attributes per person (`height_cm`, `hair_color_index`)
- Profile-aware state payloads on MQTT
- Candidate/conversation event publishing for external match authority
- No final match confirmation performed locally

In [1]:
from dataclasses import dataclass
from datetime import datetime, timezone
import json
import math
from pathlib import Path
import random
import sys
import time
from typing import Optional
import uuid

cwd = Path.cwd()
candidate_paths = [cwd / "src", cwd.parent / "src"]
for path in candidate_paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
import yaml

In [2]:
app_cfg = load_config()

config_path = Path("config.yaml")
if not config_path.exists():
    config_path = Path("../config.yaml")

raw_cfg = {}
if config_path.exists():
    raw_cfg = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}

sim_cfg = raw_cfg.get("dating_simulation", {})

RANDOM_SEED = int(sim_cfg.get("random_seed", 42))
BAR_SIZE = float(sim_cfg.get("bar_size_m", 20.0))
PEOPLE_COUNT = int(sim_cfg.get("people_count", 50))
COLORS = sim_cfg.get("colors", ["red", "blue", "green", "orange", "purple"])
CONVERSATION_RADIUS_M = float(sim_cfg.get("conversation_radius_m", 1.0))
WALK_STEP_MAX_M = float(sim_cfg.get("walk_step_max_m", 0.3))
MATCH_SECONDS = int(sim_cfg.get("match_seconds", 10))
EXIT_SECONDS = int(sim_cfg.get("exit_seconds", 20))
SIMULATION_SECONDS = int(sim_cfg.get("simulation_seconds", 300))
TICK_SECONDS = float(sim_cfg.get("tick_seconds", 1.0))
BAR_ID = str(sim_cfg.get("bar_id", "bar_1"))

HEIGHT_MIN_CM = int(sim_cfg.get("height_min_cm", 150))
HEIGHT_MAX_CM = int(sim_cfg.get("height_max_cm", 200))
HAIR_INDEX_MIN = int(sim_cfg.get("hair_index_min", 0))
HAIR_INDEX_MAX = int(sim_cfg.get("hair_index_max", 49))

TOPIC_BASE = app_cfg.mqtt.base_topic
TOPIC_STATE = f"{TOPIC_BASE}/agents/state"
TOPIC_CONVERSATION = f"{TOPIC_BASE}/events/conversation"
TOPIC_CANDIDATE = f"{TOPIC_BASE}/events/candidate"
TOPIC_MATCH = f"{TOPIC_BASE}/events/match"

random.seed(RANDOM_SEED)

print("Loaded Phase 3 settings from config.")
print(
    {
        "base_topic": TOPIC_BASE,
        "bar_id": BAR_ID,
        "people_count": PEOPLE_COUNT,
        "height_range": [HEIGHT_MIN_CM, HEIGHT_MAX_CM],
        "hair_index_range": [HAIR_INDEX_MIN, HAIR_INDEX_MAX],
        "match_seconds": MATCH_SECONDS,
        "radius_m": CONVERSATION_RADIUS_M,
        "walk_step_max_m": WALK_STEP_MAX_M,
        "simulation_seconds": SIMULATION_SECONDS,
        "tick_seconds": TICK_SECONDS,
    }
)

Loaded Phase 3 settings from config.
{'base_topic': 'simulated-city', 'bar_id': 'bar_1', 'people_count': 50, 'height_range': [150, 200], 'hair_index_range': [0, 49], 'match_seconds': 10, 'radius_m': 1.0, 'walk_step_max_m': 0.3, 'simulation_seconds': 300, 'tick_seconds': 1.0}


In [3]:
@dataclass
class Person:
    person_id: int
    color: str
    height_cm: int
    hair_color_index: int
    x: float
    y: float
    state: str = "idle"
    talking_with: Optional[int] = None
    talking_seconds: float = 0.0
    exiting_seconds: float = 0.0
    exit_dx: float = 0.0
    exit_dy: float = 0.0

def distance(a: Person, b: Person) -> float:
    return math.hypot(a.x - b.x, a.y - b.y)

def assign_exit_direction(person: Person) -> None:
    distances = {
        "left": person.x,
        "right": BAR_SIZE - person.x,
        "bottom": person.y,
        "top": BAR_SIZE - person.y,
    }
    nearest_side = min(distances, key=distances.get)

    if nearest_side == "left":
        person.exit_dx, person.exit_dy = -1.0, 0.0
    elif nearest_side == "right":
        person.exit_dx, person.exit_dy = 1.0, 0.0
    elif nearest_side == "bottom":
        person.exit_dx, person.exit_dy = 0.0, -1.0
    else:
        person.exit_dx, person.exit_dy = 0.0, 1.0

def random_walk(person: Person) -> None:
    if person.state in {"matched", "removed"}:
        return

    step_x = random.uniform(-WALK_STEP_MAX_M, WALK_STEP_MAX_M)
    step_y = random.uniform(-WALK_STEP_MAX_M, WALK_STEP_MAX_M)
    person.x = min(BAR_SIZE, max(0.0, person.x + step_x))
    person.y = min(BAR_SIZE, max(0.0, person.y + step_y))

In [4]:
def initialize_people() -> list[Person]:
    colors = (COLORS * (PEOPLE_COUNT // len(COLORS) + 1))[:PEOPLE_COUNT]
    random.shuffle(colors)

    height_pool = list(range(HEIGHT_MIN_CM, HEIGHT_MAX_CM + 1))
    if PEOPLE_COUNT > len(height_pool):
        raise ValueError("Not enough unique integer heights for people_count")
    unique_heights = random.sample(height_pool, PEOPLE_COUNT)

    hair_pool = list(range(HAIR_INDEX_MIN, HAIR_INDEX_MAX + 1))
    if PEOPLE_COUNT > len(hair_pool):
        raise ValueError("Not enough unique hair indices for people_count")
    unique_hair_indices = random.sample(hair_pool, PEOPLE_COUNT)

    people = []
    for idx in range(PEOPLE_COUNT):
        people.append(
            Person(
                person_id=idx,
                color=colors[idx],
                height_cm=unique_heights[idx],
                hair_color_index=unique_hair_indices[idx],
                x=random.uniform(0.0, BAR_SIZE),
                y=random.uniform(0.0, BAR_SIZE),
            )
        )
    return people


def clear_conversation(person: Person) -> None:
    person.talking_with = None
    person.talking_seconds = 0
    if person.state == "talking":
        person.state = "idle"


def pair_idle_people(people: list[Person]) -> None:
    available = [p for p in people if p.state == "idle"]
    random.shuffle(available)

    for person in available:
        if person.state != "idle":
            continue

        candidates = [
            other
            for other in people
            if other.person_id != person.person_id
            and other.state == "idle"
            and distance(person, other) <= CONVERSATION_RADIUS_M
        ]

        if not candidates:
            continue

        partner = random.choice(candidates)
        person.state = "talking"
        partner.state = "talking"
        person.talking_with = partner.person_id
        partner.talking_with = person.person_id

In [5]:
def update_conversations(people: list[Person]) -> tuple[int, list[dict]]:
    matched_pairs = 0
    match_events = []

    by_id = {p.person_id: p for p in people}

    for person in people:
        if person.state != "talking" or person.talking_with is None:
            continue

        partner = by_id.get(person.talking_with)
        if partner is None or partner.state != "talking" or partner.talking_with != person.person_id:
            clear_conversation(person)
            continue

        if distance(person, partner) > CONVERSATION_RADIUS_M:
            clear_conversation(person)
            clear_conversation(partner)
            continue

        if person.person_id < partner.person_id:
            person.talking_seconds += TICK_SECONDS
            partner.talking_seconds = person.talking_seconds

    return matched_pairs, match_events

def update_exits(people: list[Person]) -> int:
    removed_now = 0
    for person in people:
        if person.state != "matched":
            continue

        person.x += person.exit_dx * WALK_STEP_MAX_M
        person.y += person.exit_dy * WALK_STEP_MAX_M

        crossed_boundary = (
            person.x < 0.0
            or person.x > BAR_SIZE
            or person.y < 0.0
            or person.y > BAR_SIZE
        )
        if crossed_boundary:
            person.state = "removed"
            removed_now += 1

    return removed_now

In [6]:
connector = MqttConnector(app_cfg.mqtt, client_id_suffix="agent-bar-1")
publisher = None
match_queue = []

try:
    connector.connect()
    if connector.wait_for_connection(timeout=3.0):
        publisher = MqttPublisher(connector)
        print("MQTT connected. Publishing enabled.")
    else:
        print("MQTT not connected in time. Running local simulation only.")
except Exception as exc:
    print(f"MQTT unavailable ({exc}). Running local simulation only.")


def publish_event(topic: str, payload: dict) -> None:
    if publisher is None:
        return
    publisher.publish_json(topic, json.dumps(payload), qos=0, retain=False)


def publish_state_events(people: list[Person], second: int) -> None:
    timestamp = datetime.now(timezone.utc).isoformat()
    for person in people:
        payload = {
            "event_id": str(uuid.uuid4()),
            "person_id": person.person_id,
            "bar_id": BAR_ID,
            "color": person.color,
            "height_cm": person.height_cm,
            "hair_color_index": person.hair_color_index,
            "state": person.state,
            "x": round(person.x, 3),
            "y": round(person.y, 3),
            "timestamp": timestamp,
        }
        publish_event(TOPIC_STATE, payload)


def publish_conversation_events(people: list[Person]) -> None:
    by_id = {p.person_id: p for p in people}
    seen_pairs = set()
    timestamp = datetime.now(timezone.utc).isoformat()

    for person in people:
        if person.state != "talking" or person.talking_with is None:
            continue

        a = min(person.person_id, person.talking_with)
        b = max(person.person_id, person.talking_with)
        pair_key = (a, b)
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)

        person_a = by_id.get(a)
        person_b = by_id.get(b)
        if person_a is None or person_b is None:
            continue

        payload = {
            "event_id": str(uuid.uuid4()),
            "candidate_id": f"{a}-{b}",
            "conversation_id": f"{a}-{b}",
            "person_a_id": a,
            "person_b_id": b,
            "height_cm_a": person_a.height_cm,
            "height_cm_b": person_b.height_cm,
            "hair_color_index_a": person_a.hair_color_index,
            "hair_color_index_b": person_b.hair_color_index,
            "bar_id": BAR_ID,
            "duration_seconds": max(person_a.talking_seconds, person_b.talking_seconds),
            "distance_m": round(distance(person_a, person_b), 3),
            "timestamp": timestamp,
        }
        publish_event(TOPIC_CONVERSATION, payload)
        publish_event(TOPIC_CANDIDATE, payload)


def apply_match_queue(people: list[Person]) -> int:
    if not match_queue:
        return 0

    by_id = {p.person_id: p for p in people}
    applied = 0

    while match_queue:
        payload = match_queue.pop(0)
        try:
            person_a_id = int(payload.get("person_a_id"))
            person_b_id = int(payload.get("person_b_id"))
        except (TypeError, ValueError):
            continue

        for pid in (person_a_id, person_b_id):
            person = by_id.get(pid)
            if person is None or person.state in {"matched", "removed"}:
                continue
            person.state = "matched"
            person.talking_with = None
            person.talking_seconds = 0.0
            person.exiting_seconds = float(EXIT_SECONDS)
            assign_exit_direction(person)
            applied += 1

    return applied


if connector is not None and connector.client is not None:
    def on_message(client, userdata, message):
        if message.topic != TOPIC_MATCH:
            return
        try:
            payload = json.loads(message.payload.decode("utf-8"))
        except Exception:
            return
        if payload.get("bar_id") != BAR_ID:
            return
        match_queue.append(payload)

    connector.client.on_message = on_message
    connector.client.subscribe(TOPIC_MATCH)
    print(f"Subscribed to: {TOPIC_MATCH}")

MQTT connected. Publishing enabled.
Subscribed to: simulated-city/events/match


In [ ]:
people = initialize_people()
local_confirmed_matches = 0
applied_central_matches = 0

for second in range(1, SIMULATION_SECONDS + 1):
    tick_started = time.monotonic()

    for person in people:
        random_walk(person)

    pair_idle_people(people)
    matched_now, match_events = update_conversations(people)
    local_confirmed_matches += matched_now
    applied_central_matches += apply_match_queue(people)
    update_exits(people)

    publish_state_events(people, second)
    publish_conversation_events(people)

    if second % 20 == 0:
        state_counts = {"idle": 0, "talking": 0, "matched": 0, "removed": 0}
        for person in people:
            state_counts[person.state] = state_counts.get(person.state, 0) + 1
        print(
            f"t={second:>3}s | states={state_counts} | local_confirmed={local_confirmed_matches} | central_applied={applied_central_matches}"
        )

    elapsed = time.monotonic() - tick_started
    sleep_seconds = max(0.0, TICK_SECONDS - elapsed)
    time.sleep(sleep_seconds)

print("\nSimulation finished.")
print(f"Local confirmed matches: {local_confirmed_matches}")
print(f"Central applied matches: {applied_central_matches}")
print(f"Removed people: {sum(1 for p in people if p.state == 'removed')}")

if connector is not None:
    connector.disconnect()

t= 20s | states={'idle': 38, 'talking': 10, 'matched': 0, 'removed': 2} | local_confirmed=0 | central_applied=2
t= 40s | states={'idle': 34, 'talking': 14, 'matched': 0, 'removed': 2} | local_confirmed=0 | central_applied=2
t= 60s | states={'idle': 34, 'talking': 14, 'matched': 0, 'removed': 2} | local_confirmed=0 | central_applied=2
t= 80s | states={'idle': 36, 'talking': 12, 'matched': 0, 'removed': 2} | local_confirmed=0 | central_applied=2
t=100s | states={'idle': 40, 'talking': 8, 'matched': 0, 'removed': 2} | local_confirmed=0 | central_applied=2
t=120s | states={'idle': 38, 'talking': 8, 'matched': 1, 'removed': 3} | local_confirmed=0 | central_applied=4
t=140s | states={'idle': 36, 'talking': 10, 'matched': 0, 'removed': 4} | local_confirmed=0 | central_applied=4
